# NN module in PyTorch

In [1]:
import torch.nn as nn

In [2]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(in_features=2, out_features=3)

In [3]:
import torch
import torch.nn as nn


class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(1, 10)
        self.fc2 = nn.Linear(10, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return

In [4]:
model = SimpleNN()
print(model)

SimpleNN(
  (fc1): Linear(in_features=1, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=1, bias=True)
)


In [5]:
input_data = torch.tensor([[1.0]])
print("Input Data:", input_data)
output = model(input_data)
print("Model Output:", output)

Input Data: tensor([[1.]])
Model Output: None


# Simplicity of nn.Sequential

In [6]:
model = nn.Sequential(nn.Linear(in_features=64, out_features=128), nn.ReLU(), nn.Linear(128, 10))

# Subclassing nn.Module

In [7]:
import torch.nn.functional as F


class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
        self.layer1 = nn.Linear(64, 128)
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        return self.layer2(x)

# Iterative designs with nn.ModuleList

In [8]:
class DynamicLayerModel(nn.Module):
    def __init__(self, layer_sizes):
        super(DynamicLayerModel, self).__init__()
        self.layers = nn.ModuleList(
                [nn.Linear(in_size, out_size) for in_size, out_size in zip(layer_sizes[:-1], layer_sizes[1:])]
        )

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x

# Organizing with nn.ModuleDict

In [9]:
class DictModel(nn.Module):
    def __init__(self):
        super(DictModel, self).__init__()
        self.layers = nn.ModuleDict(
                {'linear': nn.Linear(64, 128), 'activation': nn.ReLU(), 'output': nn.Linear(128, 10)}
        )

    def forward(self, x):
        x = self.layers['activation'](self.layers['linear'](x))
        return self.layers['output'](x)

# Combining methods: Hybrid approaches

In [10]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(AutoEncoder, self).__init__()

        # Encoder defined using nn.Sequential
        self.encoder = nn.Sequential(nn.Linear(input_dim, encoding_dim), nn.ReLU())

        # Decoder defined as a custom nn.Module
        class Decoder(nn.Module):
            def __init__(self, encoding_dim, input_dim):
                super().__init__()
                self.layer = nn.Linear(encoding_dim, input_dim)

            def forward(self, x):
                return torch.sigmoid(self.layer(x))

        self.decoder = Decoder(encoding_dim, input_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return

# Nested models for modularity

In [11]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(ConvBlock, self).__init__()
        self.block = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding), nn.ReLU(), nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)


class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Nesting the ConvBlock modules
        self.conv1 = ConvBlock(3, 16, 3, 1, 1)  # Example for an image with 3 channels
        self.conv2 = ConvBlock(16, 32, 3, 1, 1)
        self.fc = nn.Linear(32 * 8 * 8, 10)  # Assuming input image size of 32x32

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Preparing for deployment: TorchScript

In [12]:
import torch
import torch.nn as nn


# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(10, 10)

    def forward(self, x):
        return self.fc1(x)


# Instantiate the model and switch to evaluation mode
model = SimpleModel()
model.eval()

# Convert the model to TorchScript using tracing
example_input = torch.randn(1, 10)
traced_script_module = torch.jit.trace(model, example_input)

# Save the TorchScript model
traced_script_module.save("simple_model.pt")
print("Model has been converted to TorchScript and saved as simple_model.pt")

Model has been converted to TorchScript and saved as simple_model.pt


# Nested models for modularity in PyTorch

In [13]:
class DoubleLinear(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DoubleLinear, self).__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))

    def forward(self, x):
        return self.layers(x)

In [14]:
class MainModel(nn.Module):
    def __init__(self):
        super(MainModel, self).__init__()
        self.block1 = DoubleLinear(10, 20, 15)
        self.block2 = DoubleLinear(15, 25, 5)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return x

In [15]:
model = MainModel()
input_data = torch.randn(1, 10)  # example input
output = model(input_data)
print(output)

tensor([[-0.1200, -0.1554, -0.2133,  0.0830, -0.0593]],
       grad_fn=<AddmmBackward0>)


# Residual Networks

In [16]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride))
        else:
            self.shortcut = nn.Sequential()


def forward(self, x):
    out = self.relu(self.conv1(x))
    out = self.conv2(out)
    out += self.shortcut(x)
    out = self.relu(out)
    return out

# Inception

In [17]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels_list):
        super(InceptionModule, self).__init__()
        self.branch1 = nn.Conv2d(in_channels, out_channels_list[0], kernel_size=1)
        self.branch2 = nn.Sequential(
                nn.Conv2d(in_channels, out_channels_list[1], kernel_size=1),
                nn.Conv2d(out_channels_list[1], out_channels_list[2], kernel_size=3, padding=1)
        )

        # ... You can add more branches for different kernel sizes.

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)

        # Concatenate outputs from all branches
        outputs = [out1, out2]
        return torch.cat(outputs, 1)


# Creating custom layers in PyTorch

In [18]:
import torch.nn as nn

In [19]:
class MyLinearLayer(nn.Module):
    def __init__(self, input_size, output_size):
        super(MyLinearLayer, self).__init__()

        # Define parameters (weights & biases)
        self.weights = nn.Parameter(torch.randn(input_size, output_size))
        self.biases = nn.Parameter(torch.randn(output_size))

    def forward(self, x):
        return torch.matmul(x, self.weights) + self.b

In [20]:
model = nn.Sequential(MyLinearLayer(784, 128), nn.ReLU(), MyLinearLayer(128, 10))

# Custom activation function

In [21]:
class ClippedReLU(nn.Module):
    def __init__(self, clip=1.0):
        super(ClippedReLU, self).__init__()
        self.clip = clip

    def forward(self, x):
        return torch.clamp(x, 0, self.clip)

# Pretrained models and PyTorch Hub

In [22]:
import torch
from torchvision import models

# Load a pre-trained version of ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Now, you can use the model for inference or further training
input_data = torch.randn(1, 3, 224, 224)  # Example input data
output = model(input_data)